## Exercise 2 — Sequence Prediction

In this exercise, the neural network receives a sequence of numbers as input and learns to predict the next value in the sequence.

We use the notation:
- $x_t$: value of the sequence at time step $t$;
- input sequence:

$$(x_1, x_2, \dots, x_n),$$
- target value:
$$x_{n+1}.$$

The goal of the model is therefore to learn the mapping

$$(x_1, x_2, \dots, x_n) \mapsto x_{n+1}.$$


# Sequence prediction

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

# Task 1: Fibonacci

We start with the Fibonacci's sequence
$$F_n = F_{n-1} + F_{n-2},$$
with
$$F_0=F_1=1.$$

In [ ]:
# We generate the Fibonacci sequence for F_n until n=49.
fib = [1, 1]
for _ in range(50):
    fib.append(fib[-1] + fib[-2])

fib = torch.tensor(fib, dtype=torch.float32)

# Input: (F_n, F_{n+1})
# Target: F_{n+2}
X = []
Y = []

for i in range(len(fib) - 2):
    X.append([fib[i], fib[i+1]])
    Y.append([fib[i+2]])

X = torch.tensor(X)
Y = torch.tensor(Y)

In [ ]:
# Model
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training
for epoch in range(5000):
    pred = model(X)

    loss = loss_fn(pred, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(epoch, loss.item())

# Test
a = torch.tensor([fib[48],fib[49]])
prediction = model(a)

print("Prediction:", prediction.item())
print("True value:", (fib[48]+fib[49]).item())
ratio = prediction.item() / (fib[48]+fib[49])

print(f"{ratio:.30}")

0 2.6953531397393875e+19
500 49411624.0
1000 1261570.125
1500 1210470.125
2000 534330.25
2500 453967.53125
3000 175996.5625
3500 1993.3873291015625
4000 1993.3873291015625
4500 1993.3873291015625
Predizione: 20365008896.0
Valore reale: 20365010944.0
0.99999988079071044921875


# Task 2: $F_n = F_{n-1} + n^k$
In the same way we can try to predict other sequences with a fixed recursive formula.

In [ ]:
# Generate the dataset

In [ ]:
# Model and training

## Question 
How does the model change increasing $k$? 

# Prime numbers

Can we predict the primes number sequence using this simple neural network?

We need a primality check on integer positive numbers.

In [34]:
def is_prime(n):
    if n < 2:
        return False

    for k in range(2, int(n**0.5) + 1):
        if n % k == 0:
            return False

    return True

Using it we can generate the first 51 prime numbers.

In [35]:
# Generate the first 51 prime numbers
primes = []

n = 2
while len(primes) < 51:
    if is_prime(n):
        primes.append(n)
    n += 1

primes = torch.tensor(primes, dtype=torch.float32)

print(primes[:50])

tensor([  2.,   3.,   5.,   7.,  11.,  13.,  17.,  19.,  23.,  29.,  31.,  37.,
         41.,  43.,  47.,  53.,  59.,  61.,  67.,  71.,  73.,  79.,  83.,  89.,
         97., 101., 103., 107., 109., 113., 127., 131., 137., 139., 149., 151.,
        157., 163., 167., 173., 179., 181., 191., 193., 197., 199., 211., 223.,
        227., 229.])


Then, we can train using the same neural network of the previous two cases. 
We don't want to use the entire sequence of prime numbers but only the first 50.

In [38]:
X = []
Y = []
for i in range(48):
    X.append([primes[i], primes[i+1]])
    Y.append([primes[i+2]])

X = torch.tensor(X)
Y = torch.tensor(Y)


# Model
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


# Training
for epoch in range(5000):
    pred = model(X)
    loss = loss_fn(pred, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(epoch, loss.item())


# Test: use the 49th and 50th primes to predict the 51st prime
a = torch.tensor([[primes[48], primes[49]]])

prediction = model(a)

print("Input primes:", primes[48].item(), primes[49].item())
print("Prediction for the 51st prime:", prediction.item())
print("Real 51st prime:", primes[50].item())

0 30593.125
500 25.370359420776367
1000 13.16705322265625
1500 13.016547203063965
2000 12.82047176361084
2500 12.573982238769531
3000 12.2725248336792
3500 11.914790153503418
4000 11.501304626464844
4500 11.039204597473145
Input primes: 227.0 229.0
Prediction for the 51st prime: 239.2025604248047
Real 51st prime: 233.0


## Questions 
Why does it not work for prime prediction?